In [5]:
import pandas as pd
df = pd.read_csv("CRMLSListing_with_rates.csv")
print(df.shape)

/var/folders/d_/gll3jlg1445_2nm3xgq_xvdc0000gn/T/ipykernel_28977/669314064.py:2: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("CRMLSListing_with_rates.csv")


(540183, 87)


Step 1  Process Date Fields

In [6]:

date_cols = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate",
]
print(df[date_cols].dtypes)
print(df[date_cols].head(3))

CloseDate                   str
PurchaseContractDate        str
ListingContractDate         str
ContractStatusChangeDate    str
dtype: object
  CloseDate PurchaseContractDate ListingContractDate ContractStatusChangeDate
0       NaN           2024-05-07          2024-01-01               2024-05-07
1       NaN                  NaN          2024-01-24               2024-01-24
2       NaN                  NaN          2024-01-12               2024-01-12


In [7]:
# Convert the date column to datetime.
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")
print(df[date_cols].dtypes)
print(df[date_cols].head(5))

CloseDate                   datetime64[us]
PurchaseContractDate        datetime64[us]
ListingContractDate         datetime64[us]
ContractStatusChangeDate    datetime64[us]
dtype: object
  CloseDate PurchaseContractDate ListingContractDate ContractStatusChangeDate
0       NaT           2024-05-07          2024-01-01               2024-05-07
1       NaT                  NaT          2024-01-24               2024-01-24
2       NaT                  NaT          2024-01-12               2024-01-12
3       NaT                  NaT          2024-01-20               2024-01-20
4       NaT                  NaT          2024-01-12               2024-01-12


In [8]:
print("CloseDate missing:", df["CloseDate"].isnull().sum())
print("PurchaseContractDate missing:", df["PurchaseContractDate"].isnull().sum())

CloseDate missing: 374361
PurchaseContractDate missing: 273091


NaT values represent missing datetime information. 
These occur when certain events, such as closing or contract signing, have not yet taken place or are not recorded. 


In [9]:
# create date consistency flags

# A. listing date is after close date
df["listing_after_close_flag"] = (
    df["ListingContractDate"].notna() &
    df["CloseDate"].notna() &
    (df["ListingContractDate"] > df["CloseDate"])
)

# B. purchase date is after close date
df["purchase_after_close_flag"] = (
    df["PurchaseContractDate"].notna() &
    df["CloseDate"].notna() &
    (df["PurchaseContractDate"] > df["CloseDate"])
)

# C. negative timeline:
# listing date is after purchase date
df["negative_timeline_flag"] = (
    df["ListingContractDate"].notna() &
    df["PurchaseContractDate"].notna() &
    (df["ListingContractDate"] > df["PurchaseContractDate"])
)
#  convert True/False to 1/0
df["listing_after_close_flag"] = df["listing_after_close_flag"].astype(int)
df["purchase_after_close_flag"] = df["purchase_after_close_flag"].astype(int)
df["negative_timeline_flag"] = df["negative_timeline_flag"].astype(int)

#  check flag counts
print("Flag counts:")
print("listing_after_close_flag:", df["listing_after_close_flag"].sum())
print("purchase_after_close_flag:", df["purchase_after_close_flag"].sum())
print("negative_timeline_flag:", df["negative_timeline_flag"].sum())
print()

Flag counts:
listing_after_close_flag: 72
purchase_after_close_flag: 265
negative_timeline_flag: 271



In [10]:
print("Rows with listing_after_close_flag = 1")
print(df.loc[df["listing_after_close_flag"] == 1,
             ["ListingContractDate", "PurchaseContractDate", "CloseDate"]].head())

print("\nRows with purchase_after_close_flag = 1")
print(df.loc[df["purchase_after_close_flag"] == 1,
             ["ListingContractDate", "PurchaseContractDate", "CloseDate"]].head())

print("\nRows with negative_timeline_flag = 1")
print(df.loc[df["negative_timeline_flag"] == 1,
             ["ListingContractDate", "PurchaseContractDate", "CloseDate"]].head())

Rows with listing_after_close_flag = 1
      ListingContractDate PurchaseContractDate  CloseDate
1406           2024-01-31           2024-01-15 2024-01-30
4140           2024-01-26           2024-01-10 2024-01-25
16407          2024-01-02           2023-12-04 2024-01-01
41873          2024-03-21           2024-03-03 2024-03-20
73360          2024-04-08           2024-03-27 2024-03-29

Rows with purchase_after_close_flag = 1
     ListingContractDate PurchaseContractDate  CloseDate
309           2024-01-31           2024-03-04 2024-01-31
919           2024-01-31           2024-02-05 2024-01-31
974           2024-01-29           2024-02-04 2024-01-29
1271          2024-01-29           2024-02-01 2024-01-29
4136          2024-01-25           2024-01-26 2024-01-25

Rows with negative_timeline_flag = 1
      ListingContractDate PurchaseContractDate  CloseDate
1406           2024-01-31           2024-01-15 2024-01-30
1712           2024-01-18           2024-01-05 2024-01-30
2665           202

In [11]:
# High-Missing Columns (≥90%)
missing_pct = df.isnull().mean() * 100
high_missing_cols = missing_pct[missing_pct >= 90].index.tolist()
print("High-missing cols:", len(high_missing_cols))

# Non-analytical Column
extra_drop = [
    "ListAgentEmail",
    "ListAgentFirstName", "ListAgentLastName", "ListAgentFullName",
    "CoListAgentFirstName", "CoListAgentLastName",
    "BuyerAgentFirstName", "BuyerAgentLastName",
    "BuyerAgentMlsId", "BuyerAgencyCompensationType",
    "ListOfficeName", "BuyerOfficeName", "CoListOfficeName",
    "BuyerOfficeAOR",
]
extra_drop = [col for col in extra_drop if col in df.columns]
print("Manual drop cols:", len(extra_drop))

before_cols = df.shape[1]
# Execute Deletion
df = df.drop(columns=high_missing_cols)
df = df.drop(columns=extra_drop)

after_cols = df.shape[1]

print("===== Check Deletion =====")
print("Before:", before_cols)
print("After:", after_cols)
print("Removed:", before_cols - after_cols)

High-missing cols: 13
Manual drop cols: 14
===== Check Deletion =====
Before: 90
After: 63
Removed: 27


In [12]:
# ---------- Duplicate Column Check ----------
# 1. Suffix duplicates (.1, .2, etc.)
suffix_dups = [col for col in df.columns if col.endswith((".1", ".2", ".3"))]

# 2. Duplicate column names
name_dups = df.columns[df.columns.duplicated()].tolist()

# 3. Duplicate content columns
content_dups = []
cols = df.columns

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        if df[cols[i]].equals(df[cols[j]]):
            content_dups.append((cols[i], cols[j]))

# ---------- Print Summary ----------
print("===== Duplicate Column Summary =====")
print(f"Suffix duplicates (.1 etc): {len(suffix_dups)}")
print(f"Duplicate column names: {len(name_dups)}")
print(f"Duplicate content pairs: {len(content_dups)}")
print("\n===== Duplicate Content Pairs =====")
for col1, col2 in content_dups:
    print(f"{col1}  <->  {col2}")


===== Duplicate Column Summary =====
Suffix duplicates (.1 etc): 11
Duplicate column names: 0
Duplicate content pairs: 8

===== Duplicate Content Pairs =====
ListingKey  <->  ListingKeyNumeric
Latitude  <->  Latitude.1
Longitude  <->  Longitude.1
UnparsedAddress  <->  UnparsedAddress.1
PropertyType  <->  PropertyType.1
LivingArea  <->  LivingArea.1
ListPrice  <->  ListPrice.1
DaysOnMarket  <->  DaysOnMarket.1


In [13]:
print("Check equality:", df["ListingKey"].equals(df["ListingKeyNumeric"]))

print("Different rows:", (df["ListingKey"] != df["ListingKeyNumeric"]).sum())

print("\nSample data:")
print(df[["ListingKey", "ListingKeyNumeric"]].head(5))

Check equality: True
Different rows: 0

Sample data:
   ListingKey  ListingKeyNumeric
0  1074973329         1074973329
1  1074954552         1074954552
2  1074936537         1074936537
3  1074917818         1074917818
4  1074143166         1074143166


In [14]:
# ---------- Drop Duplicate Columns ----------

# 1. Drop suffix duplicates (.1, .2, etc.)
df = df.drop(columns=suffix_dups, errors="ignore")
print(f"Dropped {len(suffix_dups)} suffix duplicate columns")

# 2. Drop confirmed redundant column
df = df.drop(columns=["ListingKeyNumeric"], errors="ignore")
print("Dropped ListingKeyNumeric (duplicate of ListingKey)")

Dropped 11 suffix duplicate columns
Dropped ListingKeyNumeric (duplicate of ListingKey)


In [15]:
#Check to ensure that the data type is correct
num_cols = [
    "ClosePrice",
    "ListPrice",
    "LivingArea",
    "DaysOnMarket",
    "BedroomsTotal",
    "BathroomsTotalInteger"
]

print(df[num_cols].dtypes)
for col in num_cols:
    print(f"\n--- {col} sample values ---")
    print(df[col].head(3))
for col in num_cols:
    print(f"{col} missing after conversion:", df[col].isna().sum())

ClosePrice               float64
ListPrice                float64
LivingArea               float64
DaysOnMarket               int64
BedroomsTotal            float64
BathroomsTotalInteger    float64
dtype: object

--- ClosePrice sample values ---
0   NaN
1   NaN
2   NaN
Name: ClosePrice, dtype: float64

--- ListPrice sample values ---
0    1340000.0
1    2500000.0
2    3150000.0
Name: ListPrice, dtype: float64

--- LivingArea sample values ---
0    1301.0
1    2788.0
2    3250.0
Name: LivingArea, dtype: float64

--- DaysOnMarket sample values ---
0    127
1      1
2      1
Name: DaysOnMarket, dtype: int64

--- BedroomsTotal sample values ---
0    2.0
1    4.0
2    6.0
Name: BedroomsTotal, dtype: float64

--- BathroomsTotalInteger sample values ---
0    2.0
1    3.0
2    4.0
Name: BathroomsTotalInteger, dtype: float64
ClosePrice missing after conversion: 394603
ListPrice missing after conversion: 0
LivingArea missing after conversion: 556
DaysOnMarket missing after conversion: 0
Bedrooms

Step 3: Outlier Check (Invalid Values)

In [16]:
# Invalid value flags 
# -------------------------------

df["invalid_close_price_flag"] = (df["ClosePrice"] <= 0).astype(int)
df["invalid_living_area_flag"] = (df["LivingArea"] <= 0).astype(int)
df["invalid_days_on_market_flag"] = (df["DaysOnMarket"] < 0).astype(int)
df["invalid_bedrooms_flag"] = (df["BedroomsTotal"] < 0).astype(int)
df["invalid_bathrooms_flag"] = (df["BathroomsTotalInteger"] < 0).astype(int)

# overall flag
df["invalid_numeric_flag"] = df[
    [
        "invalid_close_price_flag",
        "invalid_living_area_flag",
        "invalid_days_on_market_flag",
        "invalid_bedrooms_flag",
        "invalid_bathrooms_flag"
    ]
].max(axis=1)

# count
print("Invalid Value Counts:")
print(df[
    [
        "invalid_close_price_flag",
        "invalid_living_area_flag",
        "invalid_days_on_market_flag",
        "invalid_bedrooms_flag",
        "invalid_bathrooms_flag",
        "invalid_numeric_flag"
    ]
].sum())

Invalid Value Counts:
invalid_close_price_flag         0
invalid_living_area_flag       359
invalid_days_on_market_flag     29
invalid_bedrooms_flag            0
invalid_bathrooms_flag           0
invalid_numeric_flag           388
dtype: int64


In [17]:
# preview rows with any invalid numeric issue
invalid_rows = df[df["invalid_numeric_flag"] == 1]

print("Rows with any invalid numeric issue:")
print(invalid_rows[
    [
        "ClosePrice",
        "LivingArea",
        "DaysOnMarket",
        "BedroomsTotal",
        "BathroomsTotalInteger"
    ]
].head(10))

Rows with any invalid numeric issue:
      ClosePrice  LivingArea  DaysOnMarket  BedroomsTotal  \
162     799000.0      1284.0           -48            3.0   
166     810000.0      1164.0           -58            2.0   
873    1625000.0      2130.0            -1            4.0   
2725         NaN      1128.0           -33            2.0   
2949         NaN         0.0            91            4.0   
6717         NaN         0.0            39            4.0   
8819         NaN         0.0            95            0.0   
8852         NaN         0.0            65            0.0   
9274         NaN         0.0            70            5.0   
9431         NaN         0.0           112            6.0   

      BathroomsTotalInteger  
162                     2.0  
166                     1.0  
873                     3.0  
2725                    3.0  
2949                    5.0  
6717                    3.0  
8819                    1.0  
8852                    0.0  
9274                 

Step 4：Geographic Data Checks

In [18]:
# -------------------------------
# Geographic flags
# -------------------------------

# 1️ Missing Values
df["missing_geo_flag"] = (
    df["Latitude"].isna() | df["Longitude"].isna()
)

# 2️.Zero Values ​​(Dummy Data)
df["zero_geo_flag"] = (
    (df["Latitude"] == 0) | (df["Longitude"] == 0)
)

# 3.Incorrect longitude direction (California should be negative).
df["invalid_longitude_flag"] = (
    df["Longitude"] > 0
)

# 4️. Outside the California area (approximate range)
df["out_of_ca_flag"] = (
    (df["Latitude"] < 32) | (df["Latitude"] > 42) |
    (df["Longitude"] < -125) | (df["Longitude"] > -114)
)

# 5. Overall Objectives
df["invalid_geo_flag"] = (
    df["missing_geo_flag"] |
    df["zero_geo_flag"] |
    df["invalid_longitude_flag"] |
    df["out_of_ca_flag"]
)

# Convert to 0 / 1
geo_cols = [
    "missing_geo_flag",
    "zero_geo_flag",
    "invalid_longitude_flag",
    "out_of_ca_flag",
    "invalid_geo_flag"
]

for col in geo_cols:
    df[col] = df[col].astype(int)
print("Geographic Issues Count:")
for col in geo_cols:
    print(f"{col}: {df[col].sum()}")

Geographic Issues Count:
missing_geo_flag: 80145
zero_geo_flag: 60
invalid_longitude_flag: 85
out_of_ca_flag: 289
invalid_geo_flag: 80434
